[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/p-adema/sparse-wsi-vit/blob/longformer/notebooks/compare_chunk_alignment.ipynb)

# Old (SDPA) vs New (FlexAttention) Sparse Attention

Controlled comparison on real Camelyon embeddings:

| | Old (Longformer-style) | New (FlexAttention) |
|---|---|---|
| Kernel | `F.scaled_dot_product_attention` | `compiled_flex_attention` (TRITON) |
| Window | Token-level: `window_size` = ±W tokens | Chunk-level: `window_size` = ±W chunks |
| Masking | Manual padding + unfold + `-inf` mask | `BlockMask` + `mask_mod` |

**Matching receptive fields**: New `window_size=1, chunk_size=256` gives each patch
3 chunks = 768 patch tokens of context. Old equivalent: `window_size=384` (±384 tokens = 769).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import h5py
import pandas as pd
import time
import gc
from pathlib import Path

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device=}")

## 1. Configuration

Set paths for Snellius and model hyperparameters.

In [ ]:
# ─── Paths (adjust for Snellius) ───────────────────────────────
CSV_PATH = Path("../splits/camelyon/0/val.csv")
FEATURES_DIR = Path("../camelyon-emb/")
FEATURES_NAME = "cls_224x224"
COORDS_NAME = "coords_224x224"
LABEL_COL = "label"

# ─── Architecture ──────────────────────────────────────────────
IN_FEATURES = 1280
EMBED_DIM = 384
NUM_HEADS = 6
HEAD_DIM = EMBED_DIM // NUM_HEADS
NUM_CLS = 8
DEPTH = 6

# ─── New FlexAttention params ─────────────────────────────────
NEW_CHUNK_SIZE = 256
NEW_WINDOW_SIZE = 1
FLEX_BLOCK_SIZE = 128

# ─── Old SDPA params (matched receptive field) ────────────────
OLD_WINDOW_SIZE = (2 * NEW_WINDOW_SIZE + 1) * NEW_CHUNK_SIZE // 2
OLD_CHUNK_SIZE = 512
OLD_DILATION = 1

new_context = NUM_CLS + (2 * NEW_WINDOW_SIZE + 1) * NEW_CHUNK_SIZE
old_context = NUM_CLS + (2 * OLD_WINDOW_SIZE + 1)
print(f"New: window_size={NEW_WINDOW_SIZE} chunks of {NEW_CHUNK_SIZE} -> {new_context} tokens/query")
print(f"Old: window_size={OLD_WINDOW_SIZE} tokens -> {old_context} tokens/query")

## 2. Load Camelyon embeddings

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"CSV: {len(df)} slides")

slides = []
for _, row in df.iterrows():
    name = str(row["slidename"])
    h5_path = FEATURES_DIR / f"{name}.h5"
    if h5_path.exists() and pd.notna(row[LABEL_COL]):
        slides.append({"name": name, "path": h5_path, "label": int(row[LABEL_COL])})

print(f"Found {len(slides)} slides with H5 files")

# Load all slides into memory for benchmarking
slide_data = []
for s in slides:
    with h5py.File(s["path"], "r") as f:
        feats = torch.from_numpy(f[FEATURES_NAME][:]).float()
        coords = torch.from_numpy(f[COORDS_NAME][:]).float()
    if feats.dim() == 3:
        feats = feats.reshape(-1, IN_FEATURES)
        coords = coords.reshape(-1, 2)
    slide_data.append({
        "name": s["name"],
        "label": s["label"],
        "input": feats,
        "coords": coords,
        "num_patches": feats.shape[0],
    })

patch_counts = [s["num_patches"] for s in slide_data]
print(f"Patch counts: min={min(patch_counts)}, max={max(patch_counts)}, "
      f"median={int(np.median(patch_counts))}, mean={int(np.mean(patch_counts))}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(patch_counts, bins=30, edgecolor="black", alpha=0.7)
ax.axvline(x=np.median(patch_counts), color="red", linestyle="--",
           label=f"median={int(np.median(patch_counts))}")
ax.set_xlabel("Number of patches per slide")
ax.set_ylabel("Count")
ax.set_title("Camelyon patch count distribution")
ax.legend()
fig.tight_layout()
plt.show()

## 3. Attention pattern visualisation

Using the median-sized slide.

In [ ]:
median_idx = np.argsort(patch_counts)[len(patch_counts) // 2]
vis_slide = slide_data[median_idx]
vis_patches = vis_slide["num_patches"]
vis_seq_len = NUM_CLS + vis_patches
print(f"Visualising: {vis_slide['name']} ({vis_patches} patches, seq_len={vis_seq_len})")

# Only materialise dense mask for slides up to 2048 patches (otherwise too slow)
VIS_CAP = min(vis_patches, 2048)
vis_sl = NUM_CLS + VIS_CAP

def mask_new(q, kv):
    if q < NUM_CLS or kv < NUM_CLS:
        return True
    return abs(q // NEW_CHUNK_SIZE - kv // NEW_CHUNK_SIZE) <= NEW_WINDOW_SIZE

def mask_old(q, kv):
    if q < NUM_CLS or kv < NUM_CLS:
        return True
    q_patch = q - NUM_CLS
    kv_patch = kv - NUM_CLS
    return abs(q_patch - kv_patch) <= OLD_WINDOW_SIZE

def materialise(mask_fn, seq_len):
    m = np.zeros((seq_len, seq_len), dtype=np.float32)
    for q in range(seq_len):
        for kv in range(seq_len):
            m[q, kv] = float(mask_fn(q, kv))
    return m

dense_new = materialise(mask_new, vis_sl)
dense_old = materialise(mask_old, vis_sl)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

axes[0].imshow(dense_old, cmap="Blues", interpolation="nearest", aspect="equal")
axes[0].set_title(f"OLD (SDPA)\nwindow_size={OLD_WINDOW_SIZE} tokens")
axes[0].set_xlabel("KV position")
axes[0].set_ylabel("Q position")

axes[1].imshow(dense_new, cmap="Blues", interpolation="nearest", aspect="equal")
axes[1].set_title(f"NEW (FlexAttention)\nwindow_size={NEW_WINDOW_SIZE} chunks of {NEW_CHUNK_SIZE}")
axes[1].set_xlabel("KV position")

diff = dense_new.astype(np.int8) - dense_old.astype(np.int8)
axes[2].imshow(diff, cmap="RdBu", interpolation="nearest", aspect="equal", vmin=-1, vmax=1)
axes[2].set_title("Difference (new - old)\nblue=new adds, red=new removes")
axes[2].set_xlabel("KV position")

for ax in axes:
    ax.axhline(y=NUM_CLS - 0.5, color="green", linewidth=1, linestyle="--", alpha=0.7)
    ax.axvline(x=NUM_CLS - 0.5, color="green", linewidth=1, linestyle="--", alpha=0.7)

fig.suptitle(
    f"{vis_slide['name']}  patches={VIS_CAP}  "
    f"old_context={old_context}  new_context={new_context}",
    fontsize=11,
)
fig.tight_layout()
plt.show()

print(f"Old density: {dense_old.sum() / (vis_sl**2):.2%}")
print(f"New density: {dense_new.sum() / (vis_sl**2):.2%}")

## 4. Theoretical FLOPs across actual slide sizes

Compute attended pairs for each slide's actual patch count.

In [ ]:
def pairs_new(patch_len):
    seq_len = NUM_CLS + patch_len
    num_chunks = (seq_len + NEW_CHUNK_SIZE - 1) // NEW_CHUNK_SIZE
    cls_pairs = NUM_CLS * seq_len
    patch_pairs = 0
    for q in range(NUM_CLS, seq_len):
        q_chunk = q // NEW_CHUNK_SIZE
        lo = max(0, q_chunk - NEW_WINDOW_SIZE)
        hi = min(num_chunks - 1, q_chunk + NEW_WINDOW_SIZE)
        attended = NUM_CLS
        for c in range(lo, hi + 1):
            c_start = c * NEW_CHUNK_SIZE
            c_end = min((c + 1) * NEW_CHUNK_SIZE, seq_len)
            for kv in range(c_start, c_end):
                if kv >= NUM_CLS:
                    attended += 1
        patch_pairs += attended
    return cls_pairs + patch_pairs

def pairs_old(patch_len):
    seq_len = NUM_CLS + patch_len
    cls_pairs = NUM_CLS * seq_len
    patch_pairs = 0
    for q_p in range(patch_len):
        lo = max(0, q_p - OLD_WINDOW_SIZE)
        hi = min(patch_len - 1, q_p + OLD_WINDOW_SIZE)
        patch_pairs += NUM_CLS + (hi - lo + 1)
    return cls_pairs + patch_pairs

def pairs_dense(patch_len):
    return (NUM_CLS + patch_len) ** 2

# Sample representative patch lengths (percentiles of actual distribution)
percentiles = [10, 25, 50, 75, 90]
sample_pls = [int(np.percentile(patch_counts, p)) for p in percentiles]
# Add min and max
sample_pls = sorted(set([min(patch_counts)] + sample_pls + [max(patch_counts)]))

flops_per_pair = 2 * NUM_HEADS * HEAD_DIM * 2
results = []

for pl in sample_pls:
    d = pairs_dense(pl)
    n = pairs_new(pl)
    o = pairs_old(pl)
    results.append({"patch_len": pl, "dense": d, "new": n, "old": o})
    print(
        f"patches={pl:6d}  "
        f"dense={d:>14,}  "
        f"new={n:>12,} ({1-n/d:.1%} sparse)  "
        f"old={o:>12,} ({1-o/d:.1%} sparse)"
    )

In [ ]:
def total_gflops(pairs, patch_len):
    seq_len = patch_len + NUM_CLS
    attn = pairs * flops_per_pair * DEPTH
    proj = (2 * seq_len * EMBED_DIM * 3 * EMBED_DIM +
            2 * seq_len * EMBED_DIM * EMBED_DIM) * DEPTH
    return (attn + proj) / 1e9

pls = [r["patch_len"] for r in results]
dense_gf = [total_gflops(r["dense"], r["patch_len"]) for r in results]
new_gf = [total_gflops(r["new"], r["patch_len"]) for r in results]
old_gf = [total_gflops(r["old"], r["patch_len"]) for r in results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(pls, dense_gf, "o-", label="Dense", color="gray")
ax.plot(pls, old_gf, "^-", label=f"Old SDPA (W={OLD_WINDOW_SIZE})", color="orange")
ax.plot(pls, new_gf, "s-", label=f"New FlexAttn (W={NEW_WINDOW_SIZE} chunks)", color="blue")
ax.set_xlabel("Number of patches")
ax.set_ylabel(f"GFLOPs ({DEPTH} layers)")
ax.set_title("Total Forward FLOPs")
ax.legend()
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(pls, [d / n for d, n in zip(dense_gf, new_gf)],
        "s-", label="New vs Dense", color="blue")
ax.plot(pls, [d / o for d, o in zip(dense_gf, old_gf)],
        "^-", label="Old vs Dense", color="orange")
ax.axhline(y=1, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Number of patches")
ax.set_ylabel("FLOPs reduction factor")
ax.set_title("Theoretical Speedup over Dense")
ax.legend()
ax.set_xscale("log", base=2)
ax.grid(True, alpha=0.3)

fig.suptitle(
    f"Camelyon slides — matched receptive field: ~{new_context} tokens/query",
    fontsize=11,
)
fig.tight_layout()
plt.show()

## 5. Wall-clock timing on real slides

Benchmark forward + backward on actual Camelyon embeddings, sampling slides at
different size percentiles.

In [ ]:
from sparse_wsi_vit.models.static_sparse_attention import (
    StaticSparseViTSlideEncoder as NewEncoder,
    hilbert_sort,
)
from sparse_wsi_vit.models.static_sparse_attention_old import (
    StaticSparseViTSlideEncoder as OldEncoder,
)

assert device.type == "cuda", "Timing benchmark requires CUDA"


def benchmark(model, emb, crd, warmup=3, repeats=10):
    model.train()
    for _ in range(warmup):
        out = model(emb, crd)
        out["logits"].sum().backward()
        model.zero_grad()

    torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = model(emb, crd)
        out["logits"].sum().backward()
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
        model.zero_grad()
    return np.median(times), np.std(times)


def peak_memory_mb(model, emb, crd):
    model.eval()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    gc.collect()
    with torch.no_grad():
        _ = model(emb, crd)
    return torch.cuda.max_memory_allocated() / 1024**2


print("Benchmark functions ready.")

In [ ]:
# Pick slides at size percentiles for benchmarking
sorted_indices = np.argsort(patch_counts)
bench_percentiles = [10, 25, 50, 75, 90]
bench_indices = [sorted_indices[int(len(sorted_indices) * p / 100)] for p in bench_percentiles]

timing = {"name": [], "patch_len": [],
          "new_ms": [], "new_std": [], "new_mem": [],
          "old_ms": [], "old_std": [], "old_mem": []}

for idx in bench_indices:
    sd = slide_data[idx]
    pl = sd["num_patches"]
    emb = sd["input"].unsqueeze(0).to(device)
    crd = sd["coords"].unsqueeze(0).to(device)

    # New (FlexAttention)
    enc_new = NewEncoder(
        in_features=IN_FEATURES, out_features=1, embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS, num_layers=DEPTH, num_cls=NUM_CLS,
        window_size=NEW_WINDOW_SIZE, chunk_size=NEW_CHUNK_SIZE,
        flex_block_size=FLEX_BLOCK_SIZE,
    ).to(device)

    new_mem = peak_memory_mb(enc_new, emb, crd)
    new_med, new_std = benchmark(enc_new, emb, crd)
    del enc_new
    torch.cuda.empty_cache()
    gc.collect()

    # Old (SDPA)
    enc_old = OldEncoder(
        in_features=IN_FEATURES, out_features=1, embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS, num_layers=DEPTH, num_cls=NUM_CLS,
        window_size=OLD_WINDOW_SIZE, dilation=OLD_DILATION,
        chunk_size=OLD_CHUNK_SIZE,
    ).to(device)

    old_mem = peak_memory_mb(enc_old, emb, crd)
    old_med, old_std = benchmark(enc_old, emb, crd)
    del enc_old, emb, crd
    torch.cuda.empty_cache()
    gc.collect()

    timing["name"].append(sd["name"])
    timing["patch_len"].append(pl)
    timing["new_ms"].append(new_med)
    timing["new_std"].append(new_std)
    timing["new_mem"].append(new_mem)
    timing["old_ms"].append(old_med)
    timing["old_std"].append(old_std)
    timing["old_mem"].append(old_mem)

    print(
        f"{sd['name'][:20]:20s}  patches={pl:6d}  "
        f"new={new_med:.1f}ms ({new_mem:.0f}MB)  "
        f"old={old_med:.1f}ms ({old_mem:.0f}MB)  "
        f"speedup={old_med/new_med:.2f}x"
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

tpls = timing["patch_len"]

ax = axes[0]
ax.errorbar(tpls, timing["old_ms"], yerr=timing["old_std"],
            fmt="^-", label="Old SDPA", color="orange", capsize=3)
ax.errorbar(tpls, timing["new_ms"], yerr=timing["new_std"],
            fmt="s-", label="New FlexAttn", color="blue", capsize=3)
ax.set_xlabel("Number of patches")
ax.set_ylabel("Forward + backward (ms)")
ax.set_title("Wall-Clock Time")
ax.legend()
ax.grid(True, alpha=0.3)

speedups = [o / n for o, n in zip(timing["old_ms"], timing["new_ms"])]
ax = axes[1]
ax.plot(tpls, speedups, "s-", color="blue")
ax.axhline(y=1, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Number of patches")
ax.set_ylabel("Speedup (old / new)")
ax.set_title("FlexAttn Speedup over SDPA")
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(tpls, timing["old_mem"], "^-", label="Old SDPA", color="orange")
ax.plot(tpls, timing["new_mem"], "s-", label="New FlexAttn", color="blue")
ax.set_xlabel("Number of patches")
ax.set_ylabel("Peak memory (MB)")
ax.set_title("Inference Memory")
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle(
    f"Camelyon slides — {DEPTH} layers, embed_dim={EMBED_DIM}",
    fontsize=11,
)
fig.tight_layout()
plt.show()

## 6. Per-slide timing across full dataset

Quick single forward pass per slide to see the time distribution.

In [ ]:
enc_new = NewEncoder(
    in_features=IN_FEATURES, out_features=1, embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS, num_layers=DEPTH, num_cls=NUM_CLS,
    window_size=NEW_WINDOW_SIZE, chunk_size=NEW_CHUNK_SIZE,
    flex_block_size=FLEX_BLOCK_SIZE,
).to(device).eval()

enc_old = OldEncoder(
    in_features=IN_FEATURES, out_features=1, embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS, num_layers=DEPTH, num_cls=NUM_CLS,
    window_size=OLD_WINDOW_SIZE, dilation=OLD_DILATION,
    chunk_size=OLD_CHUNK_SIZE,
).to(device).eval()

# Warmup
warmup_emb = slide_data[0]["input"].unsqueeze(0).to(device)
warmup_crd = slide_data[0]["coords"].unsqueeze(0).to(device)
with torch.no_grad():
    _ = enc_new(warmup_emb, warmup_crd)
    _ = enc_old(warmup_emb, warmup_crd)
del warmup_emb, warmup_crd

all_pls, all_new_ms, all_old_ms = [], [], []

for sd in slide_data:
    emb = sd["input"].unsqueeze(0).to(device)
    crd = sd["coords"].unsqueeze(0).to(device)

    with torch.no_grad():
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = enc_new(emb, crd)
        torch.cuda.synchronize()
        new_ms = (time.perf_counter() - t0) * 1000

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = enc_old(emb, crd)
        torch.cuda.synchronize()
        old_ms = (time.perf_counter() - t0) * 1000

    all_pls.append(sd["num_patches"])
    all_new_ms.append(new_ms)
    all_old_ms.append(old_ms)
    del emb, crd

del enc_new, enc_old
torch.cuda.empty_cache()

all_pls = np.array(all_pls)
all_new_ms = np.array(all_new_ms)
all_old_ms = np.array(all_old_ms)

print(f"Timed {len(all_pls)} slides.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(all_pls, all_old_ms, s=10, alpha=0.5, label="Old SDPA", color="orange")
ax.scatter(all_pls, all_new_ms, s=10, alpha=0.5, label="New FlexAttn", color="blue")
ax.set_xlabel("Number of patches")
ax.set_ylabel("Inference time (ms)")
ax.set_title("Per-Slide Inference Time")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
all_speedups = all_old_ms / all_new_ms
ax.scatter(all_pls, all_speedups, s=10, alpha=0.5, color="blue")
ax.axhline(y=1, color="gray", linestyle=":", alpha=0.5)
ax.axhline(y=np.median(all_speedups), color="red", linestyle="--",
           label=f"median={np.median(all_speedups):.2f}x")
ax.set_xlabel("Number of patches")
ax.set_ylabel("Speedup (old / new)")
ax.set_title("Per-Slide Speedup")
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle(f"All {len(all_pls)} Camelyon slides (single forward pass)", fontsize=11)
fig.tight_layout()
plt.show()

print(f"Median speedup: {np.median(all_speedups):.2f}x")
print(f"Mean speedup:   {np.mean(all_speedups):.2f}x")
print(f"Total old: {all_old_ms.sum()/1000:.1f}s  new: {all_new_ms.sum()/1000:.1f}s")

## 7. Training throughput estimate

In [ ]:
ACCUMULATE_GRAD_STEPS = 8
TRAINING_ITERATIONS = 2000

# Use median slide time as representative
median_new_ms = np.median(all_new_ms)
median_old_ms = np.median(all_old_ms)
median_patches = int(np.median(all_pls))

new_step_ms = median_new_ms * ACCUMULATE_GRAD_STEPS
old_step_ms = median_old_ms * ACCUMULATE_GRAD_STEPS

new_total_h = new_step_ms * TRAINING_ITERATIONS / 1e3 / 3600
old_total_h = old_step_ms * TRAINING_ITERATIONS / 1e3 / 3600

print(f"=== Training Time Estimates ===")
print(f"Based on median slide ({median_patches} patches)")
print(f"accum={ACCUMULATE_GRAD_STEPS}, iters={TRAINING_ITERATIONS}")
print()
print(f"New FlexAttn: {new_step_ms:.0f} ms/iter -> {new_total_h:.1f}h")
print(f"Old SDPA:     {old_step_ms:.0f} ms/iter -> {old_total_h:.1f}h")
print(f"Saved:        {old_total_h - new_total_h:.1f}h ({(1 - new_total_h/old_total_h)*100:.0f}% faster)")

## 8. Summary

In [ ]:
print("=" * 60)
print("OLD (SDPA) vs NEW (FlexAttention) — CAMELYON BENCHMARK")
print("=" * 60)
print()
print(f"Dataset: {len(slide_data)} Camelyon slides")
print(f"Patches: min={min(patch_counts)}, median={int(np.median(patch_counts))}, "
      f"max={max(patch_counts)}")
print()
print(f"Architecture: {DEPTH} layers, embed_dim={EMBED_DIM}, "
      f"num_heads={NUM_HEADS}, num_cls={NUM_CLS}")
print(f"New: chunk_size={NEW_CHUNK_SIZE}, window_size={NEW_WINDOW_SIZE} chunks, "
      f"flex_block_size={FLEX_BLOCK_SIZE}")
print(f"Old: window_size={OLD_WINDOW_SIZE} tokens, dilation={OLD_DILATION}")
print(f"Context per query: new={new_context}, old={old_context}")
print()
print("--- Per-Slide Timing (all slides) ---")
print(f"Median speedup: {np.median(all_speedups):.2f}x")
print(f"Mean speedup:   {np.mean(all_speedups):.2f}x")
print(f"Total inference: old={all_old_ms.sum()/1000:.1f}s  new={all_new_ms.sum()/1000:.1f}s")
print()
print("--- Benchmarked Slides (fwd+bwd, 10 repeats) ---")
for i in range(len(timing["patch_len"])):
    sp = timing["old_ms"][i] / timing["new_ms"][i]
    print(f"  {timing['name'][i][:25]:25s}  patches={timing['patch_len'][i]:6d}  "
          f"new={timing['new_ms'][i]:.1f}ms  old={timing['old_ms'][i]:.1f}ms  {sp:.2f}x")
print()
print(f"--- Estimated Training Time ({TRAINING_ITERATIONS} iters) ---")
print(f"New FlexAttn: {new_total_h:.1f}h")
print(f"Old SDPA:     {old_total_h:.1f}h")
print(f"Saved:        {old_total_h - new_total_h:.1f}h")